# Exercise: Diagnose a Tumor — and Compare 3 Classifiers

**Scenario:** A hospital lab measures tiny features of cell nuclei from a tumor sample (radius, texture, smoothness, and so on). From those measurements, they want to predict whether a tumor is **malignant** (cancerous) or **benign** (harmless). 🏥

This is a **binary classification** problem — and you've now learned three models that can tackle it. Your job in this exercise is to train **all three, one by one**, and compare them:

1. **Logistic Regression**
2. **Support Vector Machine (SVM)**
3. **Decision Tree**

For each one, print a **classification report** so you can compare precision, recall, and f1 — not just accuracy.

The dataset is `breast_cancer.csv`. The target column is `target`: **`0` = malignant, `1` = benign**.

> Fill in the code under each step. Hints are in the comments. Try it yourself before peeking at the solution notebook!

## Step 1 — Load the data

Read `breast_cancer.csv` into a DataFrame called `df` and show the first few rows. How many rows and columns are there?

In [4]:
# import pandas as pd
import pandas as pd
# read 'breast_cancer.csv' into df
df = pd.read_csv('breast_cancer.csv')
# show df.shape and df.head()
print(df.shape)
df.head()


(569, 31)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


## Step 2 — Get to know the data

Two quick checks before modelling:
- How **balanced** is the target? (`df['target'].value_counts(normalize=True)`)
- Are there any **missing values**? (`df.isnull().sum().sum()`)

In [15]:
# check the class balance
print(df.target.value_counts(normalize=True).round(3))
# check for missing values
print("\nNull values:",df.isnull().sum().sum())

target
1    0.627
0    0.373
Name: proportion, dtype: float64

Null values: 0


## Step 3 — Features (X) and target (y)

`X` = every column **except** `target`. `y` = the `target` column.

In [16]:
# X = df.drop(columns='target')
X = df.drop(columns='target')
# y = df['target']
y = df['target']
# print X.shape
print(X.shape)

(569, 30)


## Step 4 — Train / test split

Hold back 20% for testing. Use `stratify=y` (to keep the class balance) and `random_state=42` (so your results are reproducible).

In [18]:
# from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
# print the number of training and testing rows
print(f"Training rows: {X_train.shape[0]}")
print(f"Testing rows: {X_test.shape[0]}")

Training rows: 455
Testing rows: 114


## Step 5 — Scale the features

**Logistic Regression** and **SVM** work better when features are on the same scale. Fit a `StandardScaler` on the **training data only**, then transform both train and test.

> Note: the **Decision Tree** doesn't need scaling (it splits on thresholds), so we'll feed *it* the original unscaled data later. Keep both versions around.

In [20]:
# from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler().fit(X_train)
scaler = StandardScaler().fit(X_train)
# X_train_scaled = scaler.transform(X_train)
X_train_scaled = scaler.transform(X_train)
# X_test_scaled  = scaler.transform(X_test)
X_test_scaled = scaler.transform(X_test)

## Step 6 — Model 1: Logistic Regression

Train `LogisticRegression` on the **scaled** training data, predict on the scaled test data, and **print the classification report** (use `target_names=['malignant', 'benign']`).

*Tip: pass `max_iter=5000` so it has room to converge.*

In [24]:
# from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import classification_report
from sklearn.metrics import classification_report
# create the model, fit on X_train_scaled / y_train
m1 = LogisticRegression().fit(X_train_scaled, y_train)
# predict on X_test_scaled
y_pred1 = m1.predict(X_test_scaled)
# print classification_report(y_test, y_pred, target_names=['malignant', 'benign'])
print('Logistic Regression accuracy:', round(m1.score(X_test_scaled, y_test), 3))
print(classification_report(y_test, y_pred1, target_names=['malignant', 'benign']))

Logistic Regression accuracy: 0.982
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



## Step 7 — Model 2: Support Vector Machine

Same drill with `SVC` on the **scaled** data. Print its classification report too.

In [28]:
# from sklearn.svm import SVC
from sklearn.svm import SVC
# create the model, fit on X_train_scaled / y_train
m2 = SVC(kernel='rbf')
m2.fit(X_train_scaled, y_train)
# predict, then print the classification report
y_pred2 = m2.predict(X_test_scaled)

print('SVM accuracy:', round(m2.score(X_test_scaled, y_test), 3))
print(classification_report(y_test, y_pred2, target_names=['malignant', 'benign']))

SVM accuracy: 0.982
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



## Step 8 — Model 3: Decision Tree

Train `DecisionTreeClassifier` — but on the **original, unscaled** `X_train` / `X_test` (trees don't need scaling). Use `random_state=42`. Print its classification report.

*Optional: also print the tree's **training** accuracy and compare it to the **test** accuracy. Notice anything?*

In [30]:
# from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import DecisionTreeClassifier
# create the model with random_state=42, fit on X_train / y_train (unscaled)
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train_scaled, y_train)
# predict on X_test, then print the classification report
y_pred3 = tree.predict(X_test_scaled)

print('Decision Tree Training accuracy:', round(tree.score(X_train_scaled, y_train), 3))
print('Decision Tree Testing accuracy:', round(tree.score(X_test_scaled, y_test), 3))
print(classification_report(y_test, y_pred3, target_names=['malignant', 'benign']))

Decision Tree Training accuracy: 1.0
Decision Tree Testing accuracy: 0.912
              precision    recall  f1-score   support

   malignant       0.85      0.93      0.89        42
      benign       0.96      0.90      0.93        72

    accuracy                           0.91       114
   macro avg       0.90      0.92      0.91       114
weighted avg       0.92      0.91      0.91       114



## Step 9 — Compare

**Questions:**
1. Which model had the best overall accuracy?
2. In this medical setting, a **malignant** tumor missed (predicted benign) is dangerous. Which model had the best **recall on the `malignant` class**?
3. Did the Decision Tree's training accuracy look much higher than its test accuracy? What does that suggest?

# _Your answers here..._

1.Logistic Regression and SVM are both good.
2.Logistic Regression and SVM are both good in recall of maligant class which is most important in medical applications.

## Bonus challenges 🚀

- Limit the Decision Tree with `max_depth=3` — does its **test** accuracy improve?
- Train a `RandomForestClassifier` (many trees voting) — can it beat all three?
- For the model you like best, plot a **confusion matrix** to see exactly where the mistakes land.